In [0]:
from pyspark.sql import functions as F

df_est = spark.table("workspace.default.silver_estabelecimentos")
df_emp = spark.table("workspace.default.silver_empresas")
df_nat = spark.table("workspace.default.silver_dominio_naturezas_juridicas")
df_mun = spark.table("workspace.default.silver_dominio_municipios")

# Filtra o grupo 1XXX (Administração Pública: 1015 a 1999)
df_governo = (
    df_est.alias("est")
    .join(df_emp.alias("emp"), "nr_cnpj_basico", "inner")
    .join(df_nat.alias("nat"), F.col("emp.cd_natureza_juridica") == F.col("nat.cd_natureza_juridica"), "inner")
    .join(df_mun.alias("mun"), F.col("est.cd_municipio") == F.col("mun.cd_municipio"), "left")
    .filter((F.col("emp.cd_natureza_juridica") >= 1000) & (F.col("emp.cd_natureza_juridica") < 2000))
    .select(
        F.col("est.nr_cnpj_completo").alias("nr_cnpj"),
        F.col("est.nr_cnpj_basico"),
        F.col("emp.nm_razao_social"),
        F.col("emp.cd_natureza_juridica"),
        F.col("nat.ds_natureza_juridica"),
        F.col("emp.ds_ente_federativo_responsavel"),
        F.when(F.col("emp.cd_natureza_juridica").isin(1015, 1023, 1031, 1040, 1058, 1066, 1074, 1082), "FEDERAL")
         .when(F.col("emp.cd_natureza_juridica").isin(1112, 1120, 1139, 1147, 1155), "ESTADUAL / DISTRITAL")
         .when(F.col("emp.cd_natureza_juridica").isin(1210, 1228, 1236, 1244, 1252), "MUNICIPAL")
         .otherwise("OUTROS_ENTES").alias("tp_esfera_governamental"),
        F.col("est.cd_situacao_cadastral"),
        F.col("est.dt_inicio_atividade"),
        F.col("mun.ds_municipio").alias("nm_municipio"),
        F.col("est.sg_uf"),
        F.current_timestamp().alias("dt_carga_gold")
    )
)

(
    df_governo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("sg_uf")
    .saveAsTable("workspace.default.gold_dp_governo_entes_publicos")
)
print("-> workspace.default.gold_dp_governo_entes_publicos gravada com sucesso.")

-> workspace.default.gold_dp_governo_entes_publicos gravada com sucesso.
